# MATH 307 — Applied Linear Algebra
## Computational Exploration: LU Factorization

**Explore, Compute, Discover**

This notebook accompanies Section 1.1. We translate the LU algorithm developed in the text directly into Python, test it on the worked examples, and investigate when row interchanges are required.

### 1. Implement the LU algorithm

We begin with the version that uses only row-replacement operations and does **not** perform row interchanges. Thus $L$ has 1's on its diagonal. If
$$R_i+cR_j\to R_i,$$
then the code calls $c$ the `multiplier` and stores its opposite, $-c$, in $L$.

In [ ]:
import numpy as np

def lu_factor(A):
    A = np.array(A, dtype=float)
    m, n = A.shape
    U = A.copy()
    L = np.eye(m)

    for j in range(min(m, n)):
        if U[j, j] == 0:
            raise ValueError(
                "A row interchange is required."
            )

        for i in range(j + 1, m):
            multiplier = -U[i, j] / U[j, j]
            U[i, :] = U[i, :] + multiplier * U[j, :]
            L[i, j] = -multiplier

    return L, U

### 2. Test the first worked example

For
$$A=\begin{bmatrix}1&-1&3\\3&-1&14\\-2&4&0\end{bmatrix},$$
compare the computed factors with the factors obtained by hand in the text.

In [ ]:
A = np.array([[ 1, -1,  3],
              [ 3, -1, 14],
              [-2,  4,  0]], dtype=float)

L, U = lu_factor(A)

print("L =")
print(L)
print("\nU =")
print(U)
print("\nDoes A = LU?", np.allclose(A, L @ U))

### 3. Explore

Use `lu_factor` to investigate the following questions. Run the cells, modify the matrices, and explain what you observe.

**Explore 1.** Apply the function to the $3\times4$ matrix from the preceding example:
$$A=\begin{bmatrix}2&4&6&-8\\3&1&19&3\\-4&-1&-7&4\end{bmatrix}.$$
What factors does the algorithm produce? Compare them with the unit-lower-triangular factorization in the text.

In [ ]:
A = np.array([[ 2,  4,  6, -8],
              [ 3,  1, 19,  3],
              [-4, -1, -7,  4]], dtype=float)

# Compute L and U, then verify A = LU.
L, U = lu_factor(A)
print(L)
print(U)
print(np.allclose(A, L @ U))

**Explore 2.** Try
$$A=\begin{bmatrix}0&1\\1&0\end{bmatrix}.$$
Where does the algorithm fail? How is the failure connected to the discussion of PLU factorization?

In [ ]:
A = np.array([[0, 1],
              [1, 0]], dtype=float)

lu_factor(A)

**Explore 3.** Construct other matrices for which the function stops because a row interchange is required.

In [ ]:
# Enter one of your matrices here.
A = np.array([
    # ...
], dtype=float)

# L, U = lu_factor(A)

**Explore 4.** Modify one of your examples slightly. Does the algorithm still require a row interchange? Record what you changed and what happened.

In [ ]:
# Perturb one of your matrices and experiment here.


### 4. What did we discover?

The failure above is not merely a programming problem. It reflects a mathematical limitation of this version of the LU algorithm: it does not perform row interchanges. A more robust implementation will incorporate permutation matrices and compute a **PLU factorization**.

A later numerical question is also worth keeping in mind: in floating-point computation, should a program test a computed pivot using `U[j, j] == 0`? We will return to issues like this when numerical behavior becomes part of the discussion.

## Extending the Algorithm: PLU Factorization

The function `lu_factor` stops when a zero pivot requires a row interchange. We now modify the algorithm so that, when this happens, it searches the rows below for a nonzero entry and performs the required interchange. At the same time, we record the row interchanges in a permutation matrix.

There is one important bookkeeping detail. If a row interchange occurs after some elimination steps have already been completed, the corresponding rows of the previously constructed part of \(L\) must also be interchanged. This is exactly the adjustment that appeared in the proof of the PLU-Factorization Theorem.


In [ ]:
def plu_factor(A):
    A = np.array(A, dtype=float)
    m, n = A.shape
    U = A.copy()
    L = np.eye(m)
    Q = np.eye(m)

    for j in range(min(m, n)):
        if U[j, j] == 0:
            pivot_row = None

            for k in range(j + 1, m):
                if U[k, j] != 0:
                    pivot_row = k
                    break

            if pivot_row is None:
                raise ValueError(
                    "No nonzero pivot is available in this column."
                )

            U[[j, pivot_row], :] = U[[pivot_row, j], :]
            Q[[j, pivot_row], :] = Q[[pivot_row, j], :]

            if j > 0:
                L[[j, pivot_row], :j] = L[[pivot_row, j], :j]

        for i in range(j + 1, m):
            multiplier = -U[i, j] / U[j, j]
            U[i, :] = U[i, :] + multiplier * U[j, :]
            L[i, j] = -multiplier

    P = Q.T
    return P, L, U


The matrix \(Q\) records the row interchanges, so the algorithm constructs

\[
QA=LU.
\]

Since a permutation matrix satisfies \(Q^{-1}=Q^T\), setting \(P=Q^T\) gives

\[
A=PLU.
\]

Test the function on the matrix from our PLU example.


In [ ]:
A = np.array([[0, 0, 6],
              [1, 2, 3],
              [2, 1, 4]], dtype=float)

P, L, U = plu_factor(A)

print("P =")
print(P)
print("\nL =")
print(L)
print("\nU =")
print(U)
print("\nDoes A = PLU?", np.allclose(A, P @ L @ U))


### Explore: From LU to PLU

1. Compare the matrices \(P\), \(L\), and \(U\) produced by `plu_factor` with the PLU factorization obtained earlier by hand.
2. Apply `plu_factor` to a matrix for which `lu_factor` stopped because a row interchange was required. Does the new function now produce a factorization satisfying \(A=PLU\)?
3. Compare `lu_factor` and `plu_factor`. What has changed in the algorithm, and what has stayed the same?
4. Construct another matrix that requires a row interchange and test your prediction before running the code.

The important new ingredient is the permutation matrix. The elimination step itself is still the same: we eliminate entries below each pivot and record the opposites of the multipliers in \(L\).
